# Processamento de Sinais via Transformada de Fourier Aplicado a Preços

Esta tese algorítmica apresenta o uso da filtragem de ruído de tendência e modelagem avançada (matemática contínua análoga à **Transformada de Fourier**), voltada ao ambiente caótico e discreto das cotações diárias/intraday no mercado mobiliário. Busca-se extrair as tangentes de ciclo.

O algoritmo Paparazi converte vetores interdimensionais não lineares para achar inversões de onda precisas.

In [ ]:
import pandas as pd
import numpy as np
import math

def calculate_paparazi_v4(df, mul2=1.5, sm_len=14):
    """
    Python equivalent of the Paparazi.v4 Pine Script indicator.
    Expects a DataFrame with 'Date' and 'Close' columns.
    Returns the DataFrame with the 'Paparazi_MS' main signal column 
    and the component bands.
    """
    close = df['Close'].values
    n = len(close)
    
    per = np.zeros(n)
    for i in range(1, n):
        per[i] = 100.0 * (close[i] / close[i-1] - 1.0)
        
    tu_mod = np.zeros((31, n))
    color = np.zeros((31, n), dtype=int) # 1: green, 2: red, 3: blue, 4: orange
    
    for x2 in range(1, 31):
        x = x2**mul2 + 1
        floor_x = int(math.floor(x))
        floor_x_half = int(math.floor(x/2))
        
        sum_per = np.zeros(n)
        for i in range(floor_x - 1, n):
            sum_per[i] = np.sum(per[i - floor_x + 1 : i + 1])
            
        tu_val = np.zeros(n)
        if floor_x_half > 0:
            for i in range(floor_x_half - 1, n):
                tu_val[i] = np.mean(sum_per[i - floor_x_half + 1 : i + 1])
        else:
            tu_val = sum_per.copy()
            
        tu_prev = np.roll(tu_val, 1)
        tu_prev[0] = 0
        
        for i in range(n):
            c = 0
            if abs(tu_val[i]) > abs(tu_prev[i]):
                c = 1 if tu_val[i] > 0 else 2
            else:
                c = 3 if tu_val[i] > tu_prev[i] else 4
            color[x2, i] = c
            tu_mod[x2, i] = math.atan(tu_val[i]) * 2 / math.pi
            
    s = np.zeros(n)
    for x2 in range(1, 31):
        s += tu_mod[x2]
    s = s / 2.0 + 15.0
    
    cr = np.zeros(n)
    cg = np.zeros(n)
    co = np.zeros(n)
    cb = np.zeros(n)
    
    for i in range(1, 30):
        x2 = i
        opp_x2 = 30 - i
        for t in range(n):
            c_i = color[x2, t]
            c_opp = color[opp_x2, t]
            
            is_less = (30 - (x2 + tu_mod[x2, t])) < s[t]
            
            cv_val = 0
            if is_less:
                if c_i == 3 or c_i == 1:
                    cv_val = c_i
            else:
                if c_opp == 2 or c_opp == 4:
                    cv_val = c_opp
                    
            if cv_val == 2: cr[t] += abs(tu_mod[x2, t])
            elif cv_val == 1: cg[t] += abs(tu_mod[x2, t])
            elif cv_val == 4: co[t] += abs(tu_mod[x2, t])
            elif cv_val == 3: cb[t] += abs(tu_mod[x2, t])
                
    wr = cr / 30.0
    wg = cg / 30.0
    wo = co / 30.0
    wb = cb / 30.0
    
    def sma(arr, length):
        res = np.zeros(n)
        for i in range(length - 1, n):
            res[i] = np.mean(arr[i - length + 1 : i + 1])
        return res
        
    sma_wr = sma(wr, sm_len)
    sma_wb = sma(wb, sm_len)
    sma_wg = sma(wg, sm_len)
    sma_wo = sma(wo, sm_len)
    
    rl = -1 - sma_wr
    bh = 0 + sma_wb
    gh = 1 + sma_wg
    ol = 0 - sma_wo
    
    wt = wg + wr + wo + wb
    ms = np.zeros(n)
    for t in range(n):
        if wt[t] > 0:
            ms[t] = (bh[t]*wb[t] + gh[t]*wg[t] + ol[t]*wo[t] + rl[t]*wr[t]) / wt[t]
        else:
            ms[t] = np.nan
            
    df['Paparazi_MS'] = ms
    return df

if __name__ == '__main__':
    import os
    if os.path.exists('IVVB11_2025.csv'):
        print("Reading IVVB11_2025.csv...")
        try:
            df = pd.read_csv('IVVB11_2025.csv')
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.sort_values(by='Date').reset_index(drop=True)
            df = calculate_paparazi_v4(df)
            df.to_csv('IVVB11_2025_Paparazi.csv', index=False)
            print("Successfully processed and saved to IVVB11_2025_Paparazi.csv")
        except Exception as e:
            print("Error parsing dataset:", e)
    else:
        print("CSV input file not found.")
